In [1]:
import os
import os.path as op

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from bluemath_tk.core.io import load_model
from bluemath_tk.datamining.pca import PCA
from bluemath_tk.interpolation.rbf import RBF

from utils.swash_plots import show_graph_for_different_parameters

#### Load model and results

In [2]:
swash_wrapper = load_model('outputs/hyswash/swash_model.pkl')
swash_wrapper.load_cases()

swash_output = xr.open_dataset('outputs/hyswash/output_postprocessed.nc')

depth_array=np.loadtxt('inputs/templates/hyswash/depth.bot')

mda_df = pd.read_csv('outputs/mda_df.csv')[['hs','hs_l0','swl']]

#### PCA

In [3]:
var_to_reconstruct = 'Hs'

pca = PCA()
_pcs_ds = pca.fit_transform(
    data=swash_output,
    vars_to_stack=["Hs"],
    coords_to_stack=["Xp"],
    pca_dim_for_rows="case_num",
    value_to_replace_nans={"Hs": 0.0},
)
pca.save_model(
    model_path=op.join('outputs/hyswash', f"pca_model_{var_to_reconstruct}.pkl"),
)

2026-02-24 05:28:55,039 - PCA - WARNING - Using 1200 out of 1200 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-02-24 05:28:55,040 - PCA - WARNING - Data contains NaNs.
2026-02-24 05:28:57,725 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


#### RBF 

In [4]:
rbf = RBF()
rbf.fit(
    subset_data=mda_df.iloc[swash_output["case_num"].values, :],
    target_data=pca.pcs_df,
    num_workers=24,
)
rbf.save_model(
    model_path=op.join('outputs/hyswash', f"rbf_model_{var_to_reconstruct}.pkl"),
)

In [5]:
variables_to_analyse_in_metamodel = ["hs", "hs_l0", "swl"]
lhs_parameters = {
    "num_dimensions": 3,
    "num_samples": 11000,
    "dimensions_names": variables_to_analyse_in_metamodel,
    "lower_bounds": [0.5, 0.005, 0],
    "upper_bounds": [3, 0.05, 1.5],
}

# To avoid excessive logging, you can disable the logger for RBF
rbf.logger.disabled = True
pca.logger.disabled = True

show_graph_for_different_parameters(
    pca=pca, rbf=rbf, lhs_parameters=lhs_parameters, depthfile='inputs/templates/hyswash/depth.bot'
)

interactive(children=(FloatSlider(value=1.4363502971184063, continuous_update=False, description='hs', max=3.0…

<function utils.swash_plots.show_graph_for_different_parameters.<locals>.update_plot(hs, hs_l0, swl)>